<a href="https://colab.research.google.com/github/iiitpratham9777/AI-driven-crop-disease-prediction-and-management-system/blob/main/WinCLIP_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WinCLIP — Zero-/Few-Shot Anomaly Detection (Google Colab)

This notebook runs the **WinCLIP** anomaly detection pipeline (from the `cdd` project you uploaded) on Google Colab.

**What this notebook does:**
1. Installs the Python packages the project needs
2. Unzips your `cdd.zip` project into `/content/cdd`
3. Downloads the CLIP (`ViT-B-16-plus-240`, LAION-400M) checkpoint the model needs
4. Lets you upload/point to your anomaly-detection dataset (MVTec-AD format)
5. Lets you set the run configuration (dataset name, object types, shot count)
6. Runs the evaluation loop and reports AUROC / AUPR / F1-max

> **Tip:** Go to `Runtime → Change runtime type` and select a **GPU** for much faster encoding (the original code defaults to CPU — this notebook switches it to GPU automatically when available).


## 1. Check the runtime (GPU recommended)

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — Runtime > Change runtime type > GPU for faster inference.")


Torch version: 2.9.0+cpu
CUDA available: False
No GPU detected — Runtime > Change runtime type > GPU for faster inference.


## 2. Install dependencies

These are the packages the project's code (`open_clip`, `datasets/mvtec_dataset.py`, `binary_focal_loss.py`) imports that aren't preinstalled on Colab.


In [ ]:
!pip install -q ftfy regex einops fvcore iopath psutil simplejson joblib opencv-python-headless scikit-learn scikit-image
print("Done installing dependencies.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 23.9 MB/s eta 0:00:00
Done installing dependencies.


## 3. Upload and extract your project (`cdd.zip`)

Run the cell below, then click **Choose Files** and select the `cdd.zip` you have locally (the one you uploaded to this chat).


In [ ]:
from google.colab import files
import os

print("Please select cdd.zip from your computer...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print("Uploaded:", zip_name)


Please select cdd.zip from your computer...


Saving cdd.zip to cdd.zip
Uploaded: cdd.zip


In [ ]:
import zipfile, os

EXTRACT_DIR = "/content"
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

# Find the extracted project folder (handles both "cdd.zip" -> "cdd/" and zips with a different top folder)
PROJECT_DIR = None
for name in os.listdir(EXTRACT_DIR):
    full = os.path.join(EXTRACT_DIR, name)
    if os.path.isdir(full) and os.path.exists(os.path.join(full, "main.py")):
        PROJECT_DIR = full
        break

assert PROJECT_DIR is not None, "Could not find the project folder (expected a folder containing main.py). Check the zip contents."
print("Project extracted to:", PROJECT_DIR)

%cd {PROJECT_DIR}


Project extracted to: /content/cdd
/content/cdd


## 4. Download the CLIP checkpoint

`main.py` loads weights from `vit_b_16_plus_240-laion400m_e31-8fb26589.pt`. This file is not included in the zip (it's ~600MB+), so we download it from the official `open_clip` release.


In [ ]:
import os

CKPT_NAME = "vit_b_16_plus_240-laion400m_e31-8fb26589.pt"
CKPT_URL = "https://github.com/mlfoundations/open_clip/releases/download/v0.2-weights/vit_b_16_plus_240-laion400m_e31-8fb26589.pt"

if not os.path.exists(CKPT_NAME):
    print("Downloading checkpoint (this may take a few minutes)...")
    !wget -q --show-progress {CKPT_URL} -O {CKPT_NAME}
else:
    print("Checkpoint already present.")

print("Checkpoint ready:", os.path.exists(CKPT_NAME), "-", os.path.getsize(CKPT_NAME) if os.path.exists(CKPT_NAME) else 0, "bytes")


vit_b_16_plus_240-l 100%[===================>] 794.94M  33.8MB/s    in 22s     
Checkpoint ready: True - 833559975 bytes


## 5. Provide your dataset

The dataset must follow the **MVTec-AD format**:

```
DATA_PATH/
    <datasetname>/
        <object_type_1>/
            train/
                good/
            test/
                good/
                defect_class_1/
                defect_class_2/
                ...
        <object_type_2>/
            ...
```

Pick **one** of the two options below.

**Option A — Upload a zip of your dataset** (run the next cell and select a `.zip` containing the `<datasetname>` folder).

**Option B — Use Google Drive** (uncomment and run the cell after that instead, then set `DATA_ROOT` to the path inside your Drive).


In [ ]:
# --- OPTION A: upload a dataset zip ---
'''from google.colab import files
import zipfile, os

DATA_ROOT = "/content/Dataset"
os.makedirs(DATA_ROOT, exist_ok=True)

print("Select your dataset .zip (skip/cancel this dialog if you're using Option B / Google Drive instead)")
try:
    uploaded_data = files.upload()
    for fname in uploaded_data.keys():
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall(DATA_ROOT)
        print("Extracted", fname, "into", DATA_ROOT)
except Exception as e:
    print("No dataset uploaded here (that's fine if you're using Google Drive instead):", e)

print("Contents of", DATA_ROOT, ":", os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else "missing")'''


'from google.colab import files\nimport zipfile, os\n\nDATA_ROOT = "/content/Dataset"\nos.makedirs(DATA_ROOT, exist_ok=True)\n\nprint("Select your dataset .zip (skip/cancel this dialog if you\'re using Option B / Google Drive instead)")\ntry:\n    uploaded_data = files.upload()\n    for fname in uploaded_data.keys():\n        with zipfile.ZipFile(fname, \'r\') as zf:\n            zf.extractall(DATA_ROOT)\n        print("Extracted", fname, "into", DATA_ROOT)\nexcept Exception as e:\n    print("No dataset uploaded here (that\'s fine if you\'re using Google Drive instead):", e)\n\nprint("Contents of", DATA_ROOT, ":", os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else "missing")'

In [ ]:
# --- OPTION B: use Google Drive instead ---
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = "/content/drive/MyDrive"   # <-- edit this path
print(os.listdir(DATA_ROOT))


Mounted at /content/drive
['Classroom', 'photo.jpg', 'iiser scan.pdf_merged.pdf', 'marks.odt', 'program.odt', '2wVHz1CgXifZABki4JjM_c-XTu4jHQbGKI5mf3-nyCxhHoQsnUjdso3PVayo4jAU.gdoc', 'Workbook - BOS 2.0 Practitioner Self-Paced Online Training (interactive) (1)_7b57eea2a3aaa11e25aeaa7a519c8ad0.pdf', 'Crystals.pdf', 'Crystals.gdoc', 'Scheme of work.docx', 'FRONT PAGE INTERNSHIP.docx', 'Copy of Gmail Sheets mail merge  (5).gsheet', 'Copy of Gmail Sheets mail merge  (4).gsheet', 'Copy of Gmail Sheets mail merge  (3).gsheet', 'Copy of Gmail Sheets mail merge  (2).gsheet', 'Bayesian Deep Learning_Trishit.gdoc', 'Untitled document.gdoc', 'Chapter 3: In-text Questions and Answers.gdoc', 'Chapter 4: In-text Questions and Answers.gdoc', 'Chapter 5: In-text Questions and Answers.gdoc', 'Write in brief about all these refineries.gdoc', 'Write in short about their histories.gdoc', 'Copy of Gmail Sheets mail merge  (1).gsheet', 'Copy of Gmail Sheets mail merge .gsheet', 'Google AI Studio', 'Colab No

## 6. Configure the run

Edit the values below to match your dataset:

- `DATASET_ROOT_DIR`: the folder that *contains* your dataset folder (`DATA_ROOT` from step 5 by default)
- `DATASET_NAME`: the name of the dataset subfolder
- `OBJECT_TYPES`: list of object/category subfolders to evaluate (use `"all"` as the last placeholder entry — the original code skips the final entry in the list, matching `OBJECT_TYPE[:-1]`)
- `SHOT`: `0` for zero-shot, or `1`/`2`/... for few-shot (number of reference "good" images used)


In [ ]:
DATASET_ROOT_DIR = DATA_ROOT          # from step 5; change if using Google Drive
DATASET_NAME = "agriculture"         # <-- edit: name of your dataset folder
OBJECT_TYPES = ["maize", "soybean", "wheat", "paddy", "all"]  # <-- edit: last entry is a placeholder and is skipped, like the original code
SHOT = 2                              # <-- edit: 0 = zero-shot, >0 = few-shot

DATA_DIR = os.path.join(DATASET_ROOT_DIR, DATASET_NAME)
print("Data directory:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))
if os.path.exists(DATA_DIR):
    print("Contents:", os.listdir(DATA_DIR))


Data directory: /content/drive/MyDrive/agriculture
Exists: True
Contents: ['wheat', 'maize', 'soybean', 'paddy']


## 7. Patch `OBJECT_TYPE` in the dataset module

`datasets/mvtec_dataset.py` hardcodes the list of object types. This cell rewrites that line in the file to match what you set above, then imports it.


In [ ]:
import re

dataset_file = "datasets/mvtec_dataset.py"
with open(dataset_file, "r") as f:
    content = f.read()

new_line = "OBJECT_TYPE = {}".format(OBJECT_TYPES)
content = re.sub(r"OBJECT_TYPE\s*=\s*\[[^\]]*\]", new_line, content, count=1)

with open(dataset_file, "w") as f:
    f.write(content)

print("Patched line:", new_line)


Patched line: OBJECT_TYPE = ['maize', 'soybean', 'wheat', 'paddy', 'all']


## 8. Load the model and run evaluation

This mirrors `main.py`, but:
- uses GPU automatically if available (the original script hardcodes CPU)
- loops over `OBJECT_TYPES[:-1]` exactly like `main.py`'s `__main__` block
- prints per-object-type and overall AUROC / AUPR / F1-max


In [ ]:
import sys, os, glob

matches = glob.glob('/content/**/main.py', recursive=True)
_project_dir = os.path.dirname(matches[0]) if matches else os.getcwd()
sys.path.insert(0, _project_dir)
os.chdir(_project_dir)
print("Project dir:", _project_dir)

import importlib
import sys

# make sure we (re)import the patched dataset module fresh
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("datasets"):
        del sys.modules[mod_name]

import torch
import json
import math
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

import open_clip
from binary_focal_loss import BinaryFocalLoss
from open_clip.model import get_cast_dtype
from open_clip.utils.env import checkpoint_pathmgr as pathmgr
import importlib.util
_spec = importlib.util.spec_from_file_location(
    "mvtec_dataset",
    os.path.join(_project_dir, "datasets", "mvtec_dataset.py")
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
mvtec_dataset = _mod.mvtec_dataset
OBJECT_TYPE = _mod.OBJECT_TYPE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

state_level = {
    "normal": ["{}", "flawless {}", "perfect {}", "unblemished {}",
               "{} without flaw", "{} without defect", "{} without damage"],
    "anomaly": ["damaged {}", "{} with flaw", "{} with defect", "{} with damage"]
}
template_level = [
    "a cropped photo of the {}.", "a cropped photo of a {}.",
    "a close-up photo of a {}.", "a close-up photo of the {}.",
    "a bright photo of a {}.", "a bright photo of the {}.",
    "a dark photo of a {}.", "a dark photo of the {}.",
    "a jpeg corrupted photo of a {}.", "a jpeg corrupted photo of the {}.",
    "a blurry photo of the {}.", "a blurry photo of a {}.",
    "a photo of the {}.", "a photo of a {}.",
    "a photo of a small {}.", "a photo of the small {}.",
    "a photo of a large {}.", "a photo of the large {}.",
    "a photo of a {} for visual inspection.", "a photo of the {} for visual inspection.",
    "a photo of a {} for anomaly detection.", "a photo of the {} for anomaly detection."
]

def get_texts(obj_name):
    normal_states = [s.format(obj_name) for s in state_level["normal"]]
    anomaly_states = [s.format(obj_name) for s in state_level["anomaly"]]
    normal_texts = [t.format(state) for state in normal_states for t in template_level]
    anomaly_texts = [t.format(state) for state in anomaly_states for t in template_level]
    return normal_texts, anomaly_texts

def run(config):
    tokenizer = open_clip.get_tokenizer('ViT-B-16-plus-240')
    _, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16-plus-240')

    cf = './open_clip/model_configs/ViT-B-16-plus-240.json'
    with open(cf, 'r') as f:
        model_cfg = json.load(f)
    embed_dim = model_cfg["embed_dim"]
    vision_cfg = model_cfg["vision_cfg"]
    text_cfg = model_cfg["text_cfg"]
    cast_dtype = get_cast_dtype('fp32')
    quick_gelu = False

    model = open_clip.model.WinCLIP(embed_dim, vision_cfg, text_cfg, quick_gelu, cast_dtype=cast_dtype)
    model = model.to(device)

    with pathmgr.open("./vit_b_16_plus_240-laion400m_e31-8fb26589.pt", "rb") as f:
        checkpoint = torch.load(f, map_location="cpu")
    model.load_state_dict(checkpoint, strict=False)

    obj_type = config['obj_type']
    shot = config["shot"]
    dataset = mvtec_dataset(config, config["data_dir"], mode='test', shot=shot, preprocess=preprocess)
    dataloader = DataLoader(dataset=dataset, batch_size=1, num_workers=2, shuffle=False)

    normal_texts, anomaly_texts = get_texts(obj_type.replace('_', " "))

    score_list = []
    gt_list = []
    for data in tqdm(dataloader, desc="Eval [{}]: ".format(obj_type), total=len(dataloader)):
        image, ref_list, mask, has_anomaly, indice = data

        pos_features = tokenizer(normal_texts).to(device)
        neg_features = tokenizer(anomaly_texts).to(device)
        pos_features = model.encode_text(pos_features)
        neg_features = model.encode_text(neg_features)
        pos_features /= pos_features.norm(dim=-1, keepdim=True)
        neg_features /= neg_features.norm(dim=-1, keepdim=True)
        pos_features = torch.mean(pos_features, dim=0, keepdim=True)
        neg_features = torch.mean(neg_features, dim=0, keepdim=True)
        pos_features /= pos_features.norm(dim=-1, keepdim=True)
        neg_features /= neg_features.norm(dim=-1, keepdim=True)
        text_features = torch.cat([pos_features, neg_features], dim=0)

        if isinstance(image, list):
            pred = []
            for i in range(len(image)):
                img = image[i].to(device)
                _, _, image_features = model.encode_image(img)
                image_features /= image_features.norm(dim=-1, keepdim=True)
                score = (100.0 * image_features @ text_features.T).softmax(dim=-1)
                score = score[0, 1].cpu().numpy()
                pred.append(score)
            text_probs = sum(pred) / len(pred)
        else:
            image = image.to(device)
            _, _, image_features = model.encode_image(image)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            text_probs = text_probs[0, 1].cpu().numpy()

        if shot == 0:
            score = text_probs
            score_list.append(score)
            gt_list.append(has_anomaly[0].numpy())
        else:
            img = [image] + ref_list
            flat_img = []
            for x in img:
                if isinstance(x, list):
                    flat_img.extend(x)
                else:
                    flat_img.append(x)
            img = flat_img
            if isinstance(img, list):
                if len(img) < 5:
                    while len(img) < 5:
                        img.append(img[-1])
                elif len(img) > 5:
                    img = img[:5]
            else:
                img = [img, img, img, img, img]

            vis_probs = model.forward(image=img)
            score = (vis_probs + text_probs) / 2
            if math.isinf(score):
                score = float(0)
            score_list.append(score)
            gt_list.append(has_anomaly[0].numpy())

    auroc = roc_auc_score(gt_list, score_list)
    precision, recall, _ = precision_recall_curve(gt_list, score_list)
    aupr = auc(recall, precision)
    f1_max = 0
    for threshold in np.arange(0, 1, 0.01):
        y_pred = (np.array(score_list) > threshold).astype(int)
        f1 = f1_score(gt_list, y_pred)
        if f1 > f1_max:
            f1_max = f1

    return gt_list, score_list, auroc, aupr, f1_max

print("Model/run functions ready.")


Project dir: /content/cdd
Using device: cpu
Model/run functions ready.


## 9. Run it

This loops over `OBJECT_TYPES[:-1]` (the final entry is treated as a placeholder, matching the original `main.py`) and prints AUROC / AUPR / F1-max per type and overall.


In [ ]:
# Patch dataset_init to fix missing recursive=True in glob calls
import glob as _glob
import os

def _fixed_dataset_init(self):
    if self.shot == 'zero':
        self.img_paths = []
        return
    if self.obj_type == 'all':
        if self.mode == 'train':
            self.img_paths = _glob.glob(os.path.join(self.data_dir, '**', 'train', 'good', '*.*'), recursive=True)
        else:
            self.img_paths = _glob.glob(os.path.join(self.data_dir, '**', 'test', '**', '*.*'), recursive=True)
    else:
        type_dir = os.path.join(self.data_dir, self.obj_type)
        if self.mode == 'train':
            img_dir = os.path.join(type_dir, 'train', 'good')
            self.img_paths = _glob.glob(os.path.join(img_dir, '*.*'))
        else:
            self.img_paths = _glob.glob(os.path.join(type_dir, 'test', '**', '*.*'), recursive=True)
    self.img_paths = sorted(self.img_paths)
    print(f"[dataset_init] Found {len(self.img_paths)} images for '{self.obj_type}'")

mvtec_dataset.dataset_init = _fixed_dataset_init
print("Patch applied.")

Patch applied.


In [ ]:
import os
import glob

obj_type = "maize"

print("Checking dataset...")

for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
    files = glob.glob(
        os.path.join(DATA_DIR, obj_type, "**", ext),
        recursive=True
    )
    print(ext, len(files))

print("\nSome folders:")
for root, dirs, files in os.walk(os.path.join(DATA_DIR, obj_type)):
    print(root, len(files))

Checking dataset...
*.jpg 0
*.jpeg 0
*.png 900
*.bmp 0

Some folders:
/content/drive/MyDrive/agriculture/maize 0
/content/drive/MyDrive/agriculture/maize/test 0
/content/drive/MyDrive/agriculture/maize/test/spot 10
/content/drive/MyDrive/agriculture/maize/test/sprouting 9
/content/drive/MyDrive/agriculture/maize/test/mold 9
/content/drive/MyDrive/agriculture/maize/test/mildew 18
/content/drive/MyDrive/agriculture/maize/test/multi_anomalies 14
/content/drive/MyDrive/agriculture/maize/test/insect damage 20
/content/drive/MyDrive/agriculture/maize/test/heat damage 8
/content/drive/MyDrive/agriculture/maize/test/crack 12
/content/drive/MyDrive/agriculture/maize/test/good 200
/content/drive/MyDrive/agriculture/maize/train 0
/content/drive/MyDrive/agriculture/maize/train/good 500
/content/drive/MyDrive/agriculture/maize/ground_truth 0
/content/drive/MyDrive/agriculture/maize/ground_truth/sprouting 9
/content/drive/MyDrive/agriculture/maize/ground_truth/insect damage 20
/content/drive/MyDrive

In [ ]:
np.random.seed(10)
torch.manual_seed(10)

config = {
    'datasetname': DATASET_NAME,
    'dataset_root_dir': DATASET_ROOT_DIR,
    'data_dir': DATA_DIR,
    'shot': SHOT,
}

all_auroc_list = []
all_aupr_list = []
all_f1_list = []
all_gt_list = []
all_score_list = []

with torch.no_grad():
    for obj_type in OBJECT_TYPE[:-1]:
        config['obj_type'] = obj_type
        gt_list, score_list, auroc, aupr, f1_max = run(config)
        all_auroc_list.append(auroc)
        all_aupr_list.append(aupr)
        all_f1_list.append(f1_max)
        all_gt_list += gt_list
        all_score_list += score_list
        print("Obj Type: {}, AUROC={:.4f}, AUPR={:.4f}, F1-Max={:.4f}".format(obj_type, auroc, aupr, f1_max))

print('Avg auroc: {:.4f}'.format(np.mean(all_auroc_list)))
print('Avg aupr: {:.4f}'.format(np.mean(all_aupr_list)))
print('Avg f1-max: {:.4f}'.format(np.mean(all_f1_list)))

auroc = roc_auc_score(all_gt_list, all_score_list)
precision, recall, _ = precision_recall_curve(all_gt_list, all_score_list)
aupr = auc(recall, precision)
f1_max = 0
for threshold in np.arange(0, 1, 0.01):
    y_pred = (np.array(all_score_list) > threshold).astype(int)
    f1 = f1_score(all_gt_list, y_pred)
    if f1 > f1_max:
        f1_max = f1
print("All Type: AUROC={:.4f}, AUPR={:.4f}, F1-Max={:.4f}".format(auroc, aupr, f1_max))


[dataset_init] Found 300 images for 'maize'


Eval [maize]:  18%|█▊        | 53/300 [37:01<2:53:09, 42.06s/it]

## Notes

- The original `main.py` hardcoded `device = torch.device('cpu')` and `dataset_root_dir = '../Dataset'` with `datasetname = "goji_berries"`. This notebook keeps the same logic but reads the device automatically and exposes the dataset config as editable variables in step 6.
- If you get a "no images found" message, double check your dataset folder structure matches the MVTec-AD layout described in step 5 and that `OBJECT_TYPES` matches your actual subfolder names exactly.
- `num_workers` was reduced from `8` to `2` in the DataLoader, which tends to be more stable on Colab's shared CPUs.
